# Regenerate Metrics And Reports

Rebuild aggregate metrics and confusion-matrix reports from saved 8-fold prediction CSVs.

In [5]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_05_REGENERATE_METRICS_AND_REPORTS_CELL_PROGRESS_1 = start_notebook_cell_progress('05_regenerate_metrics_and_reports.ipynb', 'Load reporting dependencies', total_steps=1)

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

mplconfig_dir = ensure_dir(ROOT / '.matplotlib')
os.environ['MPLCONFIGDIR'] = str(mplconfig_dir.resolve())

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

finish_notebook_cell_progress(NB_05_REGENERATE_METRICS_AND_REPORTS_CELL_PROGRESS_1)


[START] 05_regenerate_metrics_and_reports.ipynb | Load reporting dependencies [0/1 step] elapsed=0.0s
[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [0/1 step] elapsed=0.0s
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [1/1 step] elapsed=0.0s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | Load reporting dependencies | done [0/1 step] elapsed=0.1s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | Load reporting dependencies | done [1/1 step] elapsed=0.1s


In [6]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_05_REGENERATE_METRICS_AND_REPORTS_CELL_PROGRESS_2 = start_notebook_cell_progress('05_regenerate_metrics_and_reports.ipynb', 'Define reporting helpers', total_steps=1)

def compute_severe_error_rate(y_true: pd.Series, y_pred: pd.Series) -> float:
    total = len(y_true)
    if total == 0:
        return 0.0
    severe = 0
    for true_label, pred_label in zip(y_true.astype(str), y_pred.astype(str), strict=True):
        if (true_label, pred_label) in SEVERE_ERROR_LABEL_PAIRS:
            severe += 1
    return float(severe / total)


def summarize_prediction_frame(prediction_df: pd.DataFrame) -> dict[str, object]:
    y_true = prediction_df['true_label'].astype(str)
    y_pred = prediction_df['predicted_label'].astype(str)
    confusion = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average='macro',
        zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'severe_error_rate': compute_severe_error_rate(y_true, y_pred),
        'confusion_matrix': confusion,
    }


def save_confusion_matrix_png(confusion: np.ndarray, output_path: Path) -> None:
    figure, axis = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        confusion,
        annot=True,
        fmt='.0f',
        cmap='Blues',
        xticklabels=LABEL_ORDER,
        yticklabels=LABEL_ORDER,
        ax=axis,
    )
    axis.set_xlabel('Predicted')
    axis.set_ylabel('Actual')
    figure.tight_layout()
    figure.savefig(output_path, dpi=200)
    plt.close(figure)


def load_prediction_csvs_for_metrics(seed_metrics_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    metrics_df = pd.read_csv(seed_metrics_path)
    prediction_frames: list[pd.DataFrame] = []
    for row in iter_notebook_progress(
        metrics_df.to_dict(orient='records'),
        '05_regenerate_metrics_and_reports.ipynb | load prediction csvs',
        total=len(metrics_df),
        unit='file',
        leave=True,
    ):
        prediction_path = Path(str(row['predictions_path']))
        prediction_df = pd.read_csv(prediction_path)
        prediction_df['fold'] = str(row.get('fold', ''))
        prediction_df['seed'] = int(row.get('seed', 0))
        prediction_frames.append(prediction_df)
    if not prediction_frames:
        raise ValueError(f'No prediction CSVs were found from {seed_metrics_path}')
    return metrics_df, pd.concat(prediction_frames, ignore_index=True)


def regenerate_official_reports(seed_metrics_path: Path, output_root: Path) -> dict[str, Path]:
    print('[REPORT] Loading prediction CSVs')
    metrics_df, all_predictions_df = load_prediction_csvs_for_metrics(seed_metrics_path)
    print(f'[REPORT] Loaded {len(all_predictions_df)} predictions')
    summary_rows: list[dict[str, object]] = []
    grouped_predictions = list(all_predictions_df.groupby(['fold', 'seed']))
    print('[REPORT] Summarizing folds')
    for (fold, seed), group_df in iter_notebook_progress(
        grouped_predictions,
        '05_regenerate_metrics_and_reports.ipynb | summarize folds',
        total=len(grouped_predictions),
        unit='fold',
        leave=True,
    ):
        summary = summarize_prediction_frame(group_df)
        summary_rows.append(
            {
                'fold': fold,
                'seed': seed,
                'accuracy': summary['accuracy'],
                'macro_precision': summary['macro_precision'],
                'macro_recall': summary['macro_recall'],
                'macro_f1': summary['macro_f1'],
                'severe_error_rate': summary['severe_error_rate'],
                'prediction_count': len(group_df),
            }
        )

    fold_summary_df = pd.DataFrame(summary_rows).sort_values(['fold', 'seed']).reset_index(drop=True)
    prediction_distribution_df = (
        all_predictions_df.groupby(['predicted_label']).size().rename('count').reset_index().sort_values('predicted_label')
    )
    overall_summary = summarize_prediction_frame(all_predictions_df)
    overall_confusion_df = pd.DataFrame(overall_summary['confusion_matrix'], index=LABEL_ORDER, columns=LABEL_ORDER)
    worst_fold_df = (
        fold_summary_df.sort_values(['macro_f1', 'severe_error_rate', 'fold', 'seed']).head(1).reset_index(drop=True)
    )
    severe_error_df = fold_summary_df[['fold', 'seed', 'severe_error_rate']].copy()
    procedure_summary_df = pd.DataFrame(
        [
            {
                'input_mode': str(metrics_df['input_mode'].iloc[0]) if 'input_mode' in metrics_df.columns else INPUT_MODE,
                'fine_tune_fraction': float(metrics_df['fine_tune_fraction'].iloc[0]) if 'fine_tune_fraction' in metrics_df.columns else FINE_TUNE_FRACTION,
                'macro_f1_mean': float(fold_summary_df['macro_f1'].mean()),
                'macro_f1_std': float(fold_summary_df['macro_f1'].std(ddof=0)),
                'worst_fold_macro_f1': float(worst_fold_df.loc[0, 'macro_f1']),
                'severe_error_rate_mean': float(fold_summary_df['severe_error_rate'].mean()),
            }
        ]
    )

    fold_summary_path = output_root / 'processed_roi8_cnn_only_fold_summary.csv'
    prediction_distribution_path = output_root / 'processed_roi8_cnn_only_prediction_distribution.csv'
    overall_confusion_csv_path = output_root / 'processed_roi8_cnn_only_overall_confusion_matrix.csv'
    overall_confusion_png_path = output_root / 'processed_roi8_cnn_only_overall_confusion_matrix.png'
    worst_fold_path = output_root / 'processed_roi8_cnn_only_worst_fold_summary.csv'
    severe_error_path = output_root / 'processed_roi8_cnn_only_severe_error_summary.csv'
    procedure_summary_path = output_root / 'processed_roi8_cnn_only_procedure_summary.csv'

    print('[REPORT] Writing report artifacts')
    fold_summary_df.to_csv(fold_summary_path, index=False)
    prediction_distribution_df.to_csv(prediction_distribution_path, index=False)
    overall_confusion_df.to_csv(overall_confusion_csv_path)
    save_confusion_matrix_png(overall_summary['confusion_matrix'], overall_confusion_png_path)
    worst_fold_df.to_csv(worst_fold_path, index=False)
    severe_error_df.to_csv(severe_error_path, index=False)
    procedure_summary_df.to_csv(procedure_summary_path, index=False)
    print('[REPORT] Report artifacts written')

    return {
        'fold_summary': fold_summary_path,
        'prediction_distribution': prediction_distribution_path,
        'overall_confusion_csv': overall_confusion_csv_path,
        'overall_confusion_png': overall_confusion_png_path,
        'worst_fold_summary': worst_fold_path,
        'severe_error_summary': severe_error_path,
        'procedure_summary': procedure_summary_path,
    }

finish_notebook_cell_progress(NB_05_REGENERATE_METRICS_AND_REPORTS_CELL_PROGRESS_2)


[START] 05_regenerate_metrics_and_reports.ipynb | Define reporting helpers [0/1 step] elapsed=0.0s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | Define reporting helpers | done [0/1 step] elapsed=0.0s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | Define reporting helpers | done [1/1 step] elapsed=0.0s


In [7]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_05_REGENERATE_METRICS_AND_REPORTS_CELL_PROGRESS_3 = start_notebook_cell_progress('05_regenerate_metrics_and_reports.ipynb', 'Regenerate reports', total_steps=1)

OUTPUT_ROOT = ensure_dir(Path(str(override('EIGHTFOLD_OUTPUT_ROOT', TRAINING_OUTPUTS_ROOT / 'mobilenetv3small_8fold_processed_roi_cnn_only'))))
SEED_METRICS_PATH = Path(str(override('SEED_METRICS_PATH', OUTPUT_ROOT / 'processed_roi8_cnn_only_seed_metrics.csv')))

written_paths = regenerate_official_reports(SEED_METRICS_PATH, OUTPUT_ROOT)
print('\n'.join(f'{name}: {path}' for name, path in written_paths.items()))

finish_notebook_cell_progress(NB_05_REGENERATE_METRICS_AND_REPORTS_CELL_PROGRESS_3)


[START] 05_regenerate_metrics_and_reports.ipynb | Regenerate reports [0/1 step] elapsed=0.0s
[REPORT] Loading prediction CSVs
[START] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [0/24 file] elapsed=0.0s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [2/24 file] elapsed=0.1s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [4/24 file] elapsed=0.2s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [6/24 file] elapsed=0.3s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [8/24 file] elapsed=0.3s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [10/24 file] elapsed=0.4s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [12/24 file] elapsed=0.4s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [14/24 file] elapsed=0.5s
[RUNNING] 05_regenerate_metrics_and_reports.ipynb | load prediction csvs [16/24 file] ela